### Innitialisation

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .getOrCreate()
)

### Nettoyage du fichier

In [2]:
from pyspark.sql import functions as F

df_raw = spark.read.option("multiLine", True).json("../data/personnages_fr.json")
#On retire les premier label qui entoure les personnages
df_perso = (
    df_raw.select(F.explode("personnages").alias("p")).select("p.*")
)


#On retire les label EN et JP et doublons et ce qui nous intéresse pas
df_perso = df_perso.drop("nom_en", "nom_jp", "stats_lvl_100", "description")

def clean_slot(slot):
    s = F.col(f"competences.{slot}")
    return s.withField(
        "niveaux",
        F.transform(s["niveaux"], lambda n: n.dropFields("nom_jp", "description_jp"))
    )

slots = ["competence_1", "competence_2", "competence_3", "incarnation"]

df_perso = df_perso.withColumn(
    "competences",
    F.struct(*[clean_slot(s).alias(s) for s in slots])
)

print("Nettoyage : \n- Des labes non concerné par des personnages\n- Des partie EN et JP \n- Des description des personnages ")

Nettoyage : 
- Des labes non concerné par des personnages
- Des partie EN et JP 
- Des description des personnages 


### Transformation

In [3]:
from functools import reduce
from pyspark.sql import functions as F

slots = ["competence_1", "competence_2", "competence_3", "incarnation"]

def niveaux_du_slot(slot):
    return (
        df_perso
        .select(
            F.col("id").alias("perso_id"),
            F.lit(slot).alias("slot"),
            F.col(f"competences.{slot}.nom").alias("competence_nom"),
            F.col(f"competences.{slot}.type").alias("competence_type"),
            F.explode(f"competences.{slot}.niveaux").alias("n"),
        )
        .select("perso_id", "slot", "competence_nom", "competence_type", "n.*")
        .withColumnRenamed("id", "niveau_id")
        .withColumnRenamed("nom", "niveau_nom")
    )

# On empile les 4 slots : une ligne par niveau de compétence
df_niveaux = reduce(lambda a, b: a.unionByName(b), [niveaux_du_slot(s) for s in slots])
df_niveaux = df_niveaux.drop("nom_jp", "description_jp")

# Découpage de la description en liste d'effets
SPLIT = r"(?<=\.)\s+(?=[A-ZÀÂÉÈÊÎÔÛÇ])"
df_niveaux = (
    df_niveaux
    .withColumn("effets", F.split(F.trim("description"), SPLIT))
    .drop("description")
)

print("Transformation : \n- Des compétences -> les effets se trouvent dans une liste ")
df_niveaux.select("perso_id", "slot", "niveau", "effets").show(3, truncate=False)

Transformation : 
- Des compétences -> les effets se trouvent dans une liste 
+---------+------------+------+-------------------------------------------------------------------------------+
|perso_id |slot        |niveau|effets                                                                         |
+---------+------------+------+-------------------------------------------------------------------------------+
|130114001|competence_1|1     |[Attaque tranchante sur un ennemi (énorme)., Recollection Field (ténèbres) +1.]|
|130114001|competence_1|2     |[Attaque tranchante sur un ennemi (énorme)., Recollection Field (ténèbres) +1.]|
|130114001|competence_1|3     |[Attaque tranchante sur un ennemi (énorme)., Recollection Field (ténèbres) +1.]|
+---------+------------+------+-------------------------------------------------------------------------------+
only showing top 3 rows


### Découpage entre tout les perso, perso 100 et perso 120

In [4]:
'''# ---------- Stats par niveau de personnage ----------
COLS_PERSO = ["id", "label", "nom", "titre", "element", "arme", "rarete", "regen_mp_taux"]

def stats_a_niveau(niv):
    return (
        df_perso
        .select(*COLS_PERSO, f"stats.lvl_{niv}.*")
        .withColumn("niveau_perso", F.lit(niv))
    )

df_perso_100 = stats_a_niveau(100)
df_perso_120 = stats_a_niveau(120)

# ---------- Compétences : règle du niveau 100 ----------
REGLE_100 = {"competence_1" : 3, "competence_2" : 3, "competence_3" : 2, "incarnate" : 3}
regle_100 = spark.createDataFrame(list(REGLE_100.items()), ["slot", "niveau"])

df_comp_100 = (
    df_niveaux
    .join(regle_100, on = ["slot", "niveau"], how = "inner")
    .withColumn("niveau_perso", F.lit(100))
)

# ---------- Compétences : niveau max (lvl 120) ----------
niv_max = df_niveaux.groupBy("perso_id", "slot").agg(F.max("niveau").alias("niveau"))

df_comp_120 = (
    df_niveaux
    .join(niv_max, on=["perso_id", "slot", "niveau"], how="inner")
    .withColumn("niveau_perso", F.lit(120))
)


for nom, df, attendu, cle in [
    ("perso_100", df_perso_100, 218, ["id"]),
    ("perso_120", df_perso_120, 218, ["id"]),
    ("comp_100",  df_comp_100,  872, ["perso_id", "slot"]),
    ("comp_120",  df_comp_120,  872, ["perso_id", "slot"]),
]:
    n = df.count()
    n_uniques = df.select(*cle).distinct().count()
    print(nom, n, n_uniques, "OK" if n == attendu == n_uniques else "PROBLEME")'''

'# ---------- Stats par niveau de personnage ----------\nCOLS_PERSO = ["id", "label", "nom", "titre", "element", "arme", "rarete", "regen_mp_taux"]\n\ndef stats_a_niveau(niv):\n    return (\n        df_perso\n        .select(*COLS_PERSO, f"stats.lvl_{niv}.*")\n        .withColumn("niveau_perso", F.lit(niv))\n    )\n\ndf_perso_100 = stats_a_niveau(100)\ndf_perso_120 = stats_a_niveau(120)\n\n# ---------- Compétences : règle du niveau 100 ----------\nREGLE_100 = {"competence_1" : 3, "competence_2" : 3, "competence_3" : 2, "incarnate" : 3}\nregle_100 = spark.createDataFrame(list(REGLE_100.items()), ["slot", "niveau"])\n\ndf_comp_100 = (\n    df_niveaux\n    .join(regle_100, on = ["slot", "niveau"], how = "inner")\n    .withColumn("niveau_perso", F.lit(100))\n)\n\n# ---------- Compétences : niveau max (lvl 120) ----------\nniv_max = df_niveaux.groupBy("perso_id", "slot").agg(F.max("niveau").alias("niveau"))\n\ndf_comp_120 = (\n    df_niveaux\n    .join(niv_max, on=["perso_id", "slot", "nive

In [5]:
print(f"Il y a actuellement {df_perso.count()} personnages")
df_perso.groupBy("nom").count().orderBy(F.desc("count")).show()
df_perso.groupBy("element").count().orderBy(F.desc("count")).show()

Il y a actuellement 218 personnages
+-------------+-----+
|          nom|count|
+-------------+-----+
|        Asuna|   26|
|       Kirito|   22|
|        Alice|   21|
|        Eydis|   16|
|        Sinon|   14|
|        Leafa|   14|
|        Eugeo|   11|
|        Tizée|   10|
|        Lonie|    8|
|        Yuuki|    8|
|Administrator|    6|
|    Soltirina|    5|
|      Lisbeth|    5|
|       Silica|    5|
|       Scheta|    4|
|       Iskahn|    3|
|          Yui|    3|
|         YUNA|    2|
|       Suguha|    2|
|     Bercouli|    2|
+-------------+-----+
only showing top 20 rows
+--------+-----+
| element|count|
+--------+-----+
|   aucun|   37|
| lumière|   33|
|ténèbres|   32|
|     feu|   31|
|   terre|   30|
|    vent|   30|
|     eau|   25|
+--------+-----+



In [6]:
#Ta question : combien de persos ont du « Soin HP sur tous les alliés » ?
(
    df_niveaux
    .filter(F.exists("effets", lambda e: e.startswith("Soins HP sur tous les alliés")))
    .select("perso_id").distinct()
    .count()
)   # 53


53

In [7]:
(
    df_niveaux
    .select("perso_id", F.explode("effets").alias("effet"))
    .groupBy("effet")
    .agg(F.countDistinct("perso_id").alias("nb_persos"))
    .orderBy(F.desc("nb_persos"))
    .show(20, truncate=80)
)

+--------------------------------------------------------------------------------+---------+
|                                                                           effet|nb_persos|
+--------------------------------------------------------------------------------+---------+
|                                                    Charge la Volonté de 20,00%.|      101|
|                                                    Charge la Volonté de 21,50%.|      101|
|                                                    Charge la Volonté de 18,50%.|      100|
|                                                     Brise la Volonté de 47,50%.|       91|
|                                                     Brise la Volonté de 50,00%.|       88|
|                                                     Brise la Volonté de 52,50%.|       85|
|                Temps de préparation court après déclenchement de la compétence.|       83|
|                                       Attaque tranchante sur un enne